# CS-781 Phase 6 - Abstractive Summarisation: all experiments in one notebook

**Setup on Kaggle:** Accelerator *GPU T4 x2*, Internet on, secret `HF` (your Hugging Face token), and the same four inputs as Phases 4-5:
`export_gqa`, `export_mla`, `export_retnet` (datasets) and the `Phase1-Arch3` notebook output (contains `export_encdec`).
The smoke check passed (29 Oct), so `SMOKE = False` is set: use *Save Version -> Save & Run All* (~6.5 h).

**How each backbone summarises**
* **Decoders (GQA, MLA, RetNet)** - one 2048-token window: `[document head ... document tail] + " Summary of the judgment:" + summary + <eos>`.
  The document part is 1152 tokens (first ~960 + last 192 when the document is longer); the loss is on the summary tokens only (up to 880).
* **Encoder-decoder** - *fusion-in-decoder*: the document is cut into up to 4 chunks of 1024 tokens (first 3 + the final chunk), each chunk is
  encoded on its own with a padding mask, and the decoder cross-attends to all 4096 encoder states. The encoder never sees positions it was not
  pre-trained on, yet the model reads 3.5x more of the judgment than the decoders. An ablation run uses a single 1024-token window.
* **Generation** - greedy decoding with n-gram blocking (fixes the looping seen in Phase 1), optionally a repetition penalty, and a per-document
  length window from a length model fitted on the training split (log summary length vs log document length). Three decoding settings are compared
  on held-out documents and the best one is used for that run.
* **Held-out split** - Phase 6 has no dev set, so ~5% of the training documents (max 300, chosen by id hash, identical for every run) are held out.
  Each run reports BERTScore-F1 / ROUGE-1 / ROUGE-L on 120 of them; the official `evaluate_phase6.py` is re-run on those predictions at the end.
  Note: BERTScore uses roberta-large, which reads at most 512 sub-word tokens, so the primary metric only sees the first ~350-400 words.

| GPU 0 | GPU 1 |
|---|---|
| LoRA - GQA (also the KD teacher) | **LoRMA** (compulsory) - enc-dec |
| QLoRA (NF4 base weights, from scratch) - RetNet | Houlsby adapters - enc-dec |
| Pfeiffer adapters - MLA | AdaLoRA - MLA |
| **knowledge distillation**: 8-layer GQA student, token-level KL to the LoRA-GQA teacher + gold CE | ablation: LoRMA enc-dec with one 1024-token window |

then a **consensus (MBR) ensemble**: per document, the run summary that agrees most with the other runs' summaries.
Every run writes `sm_<method>_<arch>/submission.zip`, `results.json` and its held-out predictions; all zips are collected in `/kaggle/working/submissions/`.
All adaptation methods (incl. NF4 quantisation) are written from scratch in `sm_peft.py`; no `peft` / `bitsandbytes`.

In [1]:
import os
HF_TOKEN = None
for _name in ("HF", "HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret(_name)
        if HF_TOKEN:
            break
    except Exception as e:
        print("secret", _name, "not available:", type(e).__name__)
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")
else:
    print("No HF token found")
import torch
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.
GPUs: ['Tesla T4', 'Tesla T4']


In [2]:
%%writefile sm_metrics.py
"""Phase 6 metrics - no torch needed.
ROUGE-1 / ROUGE-L F1 computed exactly like evaluate_phase6.py: rouge_score's DefaultTokenizer (lower-case, non-alphanumerics
removed, Porter stemmer for tokens longer than 3 characters), unigram-overlap ROUGE-1 and a bit-parallel LCS for ROUGE-L,
averaged over documents. BERTScore F1 = bert_score with lang='en' (roberta-large, no idf, no baseline rescaling), as in the
evaluator. Note: roberta-large reads at most 512 sub-word tokens, so BERTScore only sees roughly the first 350-400 words of a
summary and of the reference - the start of a summary matters most for the primary metric.
Also: a length model (how long should the summary of this document be), MBR selection for the ensemble, redundancy stats."""
import re, json, zipfile
from collections import Counter
import numpy as np

try:
    from rouge_score import tokenizers as _rt
    _TOK = _rt.DefaultTokenizer(use_stemmer=True)
    ROUGE_IMPL = "rouge_score"

    def rtok(s):
        return _TOK.tokenize(s)
except Exception:
    try:
        from nltk.stem import porter as _porter
        _st = _porter.PorterStemmer()
        _stem = lambda w: _st.stem(w) if len(w) > 3 else w
        ROUGE_IMPL = "fallback (nltk Porter)"
    except Exception:
        _stem = lambda w: w
        ROUGE_IMPL = "fallback (no stemming)"

    def rtok(s):
        return [_stem(w) for w in re.sub(r"[^a-z0-9]+", " ", s.lower()).split()]


def lcs(a, b):
    """Bit-parallel LCS length, identical to evaluate_phase6.py."""
    if len(a) > len(b):
        a, b = b, a
    masks = {}
    for i, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << i)
    s = 0
    for t in b:
        m = masks.get(t, 0)
        x = m | s
        y = (s << 1) | 1
        s = x & ~(x - y)
    return bin(s).count("1")


def _f(p, r):
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def rouge_pair(pred, ref):
    pt, rt = rtok(pred), rtok(ref)
    if not pt and not rt:
        return 1.0, 1.0
    if not pt or not rt:
        return 0.0, 0.0
    ov = sum((Counter(pt) & Counter(rt)).values())
    r1 = _f(ov / len(pt), ov / len(rt))
    z = lcs(pt, rt)
    return r1, _f(z / len(pt), z / len(rt))


def rouge(preds, refs):
    s = np.array([rouge_pair(p, r) for p, r in zip(preds, refs)])
    return float(s[:, 0].mean()), float(s[:, 1].mean())


_BS = [None, None]


def bertscore(preds, refs, device=None):
    """Mean BERTScore F1, or None when bert_score / roberta-large is not available."""
    try:
        if _BS[0] is None:
            from bert_score import BERTScorer
            _BS[0] = BERTScorer(lang="en", device=device)
        return float(_BS[0].score(list(preds), list(refs), batch_size=8)[2].mean())
    except Exception as e:
        if _BS[1] is None:
            print("BERTScore unavailable:", type(e).__name__, str(e)[:200])
            _BS[1] = True
        return None


def redundancy(text, n=4):
    """Share of n-grams that repeat an earlier n-gram of the same summary (0 = no repetition)."""
    w = text.lower().split()
    g = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return 0.0 if not g else 1.0 - len(set(g)) / len(g)


def evaluate_texts(preds, refs, device=None, bert=True):
    r1, rl = rouge(preds, refs)
    out = dict(rouge1=r1, rougeL=rl, bertscore=bertscore(preds, refs, device) if bert else None,
               words_pred=float(np.mean([len(p.split()) for p in preds])), words_ref=float(np.mean([len(r.split()) for r in refs])),
               redundancy4=float(np.mean([redundancy(p) for p in preds])), n=len(preds))
    out["score"] = out["bertscore"] if out["bertscore"] is not None else out["rouge1"]
    return out


def fmt(m):
    bs = "n/a" if m.get("bertscore") is None else f"{m['bertscore']:.4f}"
    return (f"BERTScore-F1 {bs} | ROUGE-1 {m['rouge1']:.4f} | ROUGE-L {m['rougeL']:.4f} | words pred {m['words_pred']:.0f} "
            f"ref {m['words_ref']:.0f} | repeated 4-grams {100 * m['redundancy4']:.1f}% | n={m['n']}")


class LengthModel:
    """log(summary words) = a * log(document words) + b, fitted on the training split; prediction clipped to the
    5th-95th percentile of training summary lengths. tpw = summary tokens per word for this tokenizer."""

    def __init__(self, doc_words, sum_words, sum_toks=None):
        dw, sw = np.maximum(np.asarray(doc_words, float), 1), np.maximum(np.asarray(sum_words, float), 1)
        self.a, self.b = np.polyfit(np.log(dw), np.log(sw), 1)
        self.lo, self.hi = np.percentile(sw, [5, 95])
        self.corr = float(np.corrcoef(np.log(dw), np.log(sw))[0, 1]) if len(dw) > 2 else 0.0
        self.tpw = float(np.sum(sum_toks) / np.sum(sw)) if sum_toks is not None else 1.3
        pred = np.array([self.words(x) for x in dw])
        self.mape = float(np.mean(np.abs(pred - sw) / sw))

    def words(self, dw):
        return float(np.clip(np.exp(self.a * np.log(max(dw, 1)) + self.b), self.lo, self.hi))

    def tokens(self, dw):
        return self.words(dw) * self.tpw

    def describe(self):
        return (f"summary words = exp({self.a:.3f} * ln(doc words) + {self.b:.3f}), clipped to [{self.lo:.0f}, {self.hi:.0f}] | "
                f"log-log corr {self.corr:.3f} | mean abs. error {100 * self.mape:.0f}% | {self.tpw:.2f} tokens/word")


def _bag(t):
    w = rtok(t)
    return Counter(w), len(w)


def mbr_select(cands):
    """cands: list over runs of summaries for ONE document. Returns the index of the candidate with the highest mean
    ROUGE-1 F1 against the other candidates (minimum Bayes risk / consensus selection)."""
    if len(cands) == 1:
        return 0
    bags = [_bag(c) for c in cands]
    best, bi = -1.0, 0
    for i, (ci, li) in enumerate(bags):
        s = 0.0
        for j, (cj, lj) in enumerate(bags):
            if i != j and li and lj:
                ov = sum((ci & cj).values())
                s += _f(ov / li, ov / lj)
        if s > best:
            best, bi = s, i
    return bi


def write_prediction(ids, texts, json_path, zip_path):
    """List format [{"id": ..., "summary": ...}], one object per id, prediction.json at the zip root."""
    out = []
    for i in ids:
        t = " ".join(str(texts[i]).split())
        assert t, f"empty summary for {i}"
        out.append({"id": i, "summary": t})
    assert len({o["id"] for o in out}) == len(out), "duplicate ids"
    json.dump(out, open(json_path, "w", encoding="utf-8"), ensure_ascii=False)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(json_path, arcname="prediction.json")
    return out


Writing sm_metrics.py


In [3]:
%%writefile sm_lib.py
"""CS-781 Phase 6 library: abstractive summarisation of long legal documents with the Phase-1 models.

decoder-only (gqa / mla / retnet), one 2048-token window:
    [document head (+ " [...] " + document tail)] + " Summary of the judgment:" + summary + <eos>
    loss on the summary tokens only; generation with the model's own KV cache / retention state.
encoder-decoder (encdec): the document is cut into up to K chunks of 1024 tokens (the first K-1 chunks + the document's
    final chunk). Each chunk is encoded separately (bidirectional, padding-masked); the encoder states of all chunks are
    concatenated and the decoder cross-attends to all of them (fusion-in-decoder), so it reads up to K x 1024 tokens of
    the document while the encoder never sees positions beyond what it was pre-trained on.
Generation: greedy decoding with n-gram blocking, optional repetition penalty, and a per-document length window
(min/max new tokens) from a length model fitted on the training split."""
import os, sys, json, math, time, random, hashlib, types, importlib.util, contextlib
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

EOS_ID, PAD_ID, BOS_ID = 0, 1, 2
SEP_TEXT = " Summary of the judgment:"
MARK_TEXT = " [...]"


# ====================================================================== Phase-1 models
def load_module_from_file(path, name):
    spec = importlib.util.spec_from_file_location(name, str(path))
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


def file_hash(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()[:10]


def find_p1_export(arch, roots=("/kaggle/input", "/kaggle/working")):
    for r in roots:
        for root, dirs, files in os.walk(r, followlinks=True):
            if "config.json" in files and "model.pt" in files:
                try:
                    if json.load(open(os.path.join(root, "config.json"))).get("arch") == arch:
                        return Path(root)
                except Exception:
                    pass
    return None


def load_p1(export_dir):
    export_dir = Path(export_dir)
    cj = json.load(open(export_dir / "config.json"))
    mod = load_module_from_file(export_dir / "legal_arch.py", f"legal_arch_{cj['arch']}_{file_hash(export_dir / 'legal_arch.py')}")
    bb = mod.build_model(mod.ArchConfig(**cj["cfg"]))
    bb.load_state_dict(torch.load(export_dir / "model.pt", map_location="cpu", weights_only=False))
    return mod, cj["cfg"], bb


def blocks_all(bb):
    return list(bb.blocks) if bb.kind == "decoder" else list(bb.enc_blocks) + list(bb.dec_blocks)


def make_student_bb(mod, cfg, full_bb, keep):
    """Decoder student: the teacher architecture with only the layers in `keep`, initialised from those layers."""
    assert full_bb.kind == "decoder"
    cfg = dict(cfg, n_layers=len(keep))
    st = mod.build_model(mod.ArchConfig(**cfg))
    st.tok_emb.load_state_dict(full_bb.tok_emb.state_dict())
    st.norm_f.load_state_dict(full_bb.norm_f.state_dict())
    for i, j in enumerate(keep):
        st.blocks[i].load_state_dict(full_bb.blocks[j].state_dict())
    return st, cfg


def count_params(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)


def max_positions(bb):
    return int(bb.rope_cos.shape[0])


# ---- padding masks for the enc-dec (the Phase-1 enc-dec never saw padding: encoder self-attention and decoder
# ---- cross-attention get key-padding masks through these module-level slots)
_ENC_MASK, _X_MASK = [None], [None]


def _masked_enc_attn_forward(self, x, cos, sin, cache=None):
    arch = sys.modules[type(self).__module__]
    B, T, _ = x.shape
    q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)
    k = self.wk(x).view(B, T, self.nh, self.hd).transpose(1, 2)
    v = self.wv(x).view(B, T, self.nh, self.hd).transpose(1, 2)
    q = arch.apply_rope(q, cos[:T], sin[:T])
    k = arch.apply_rope(k, cos[:T], sin[:T])
    m = _ENC_MASK[0]
    o = F.scaled_dot_product_attention(q, k, v, attn_mask=None if m is None else m[:, None, None, :T])
    return self.wo(o.transpose(1, 2).reshape(B, T, -1))


def _masked_cross_forward(self, x, enc_out, cache=None):
    B, T, _ = x.shape
    q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)
    if cache is not None and "k" in cache:
        k, v = cache["k"], cache["v"]
    else:
        Se = enc_out.shape[1]
        k = self.wk(enc_out).view(B, Se, self.nh, self.hd).transpose(1, 2)
        v = self.wv(enc_out).view(B, Se, self.nh, self.hd).transpose(1, 2)
        if cache is not None:
            cache["k"], cache["v"] = k, v
    m = _X_MASK[0]
    o = F.scaled_dot_product_attention(q, k, v, attn_mask=None if m is None else m[:, None, None, :k.shape[2]])
    return self.wo(o.transpose(1, 2).reshape(B, T, -1))


def patch_encdec(bb):
    """Must run before adapters wrap the attention modules."""
    if bb.kind != "encdec":
        return 0
    for blk in bb.enc_blocks:
        blk.attn.forward = types.MethodType(_masked_enc_attn_forward, blk.attn)
    for blk in bb.dec_blocks:
        blk.cross_attn.forward = types.MethodType(_masked_cross_forward, blk.cross_attn)
    return len(bb.enc_blocks) + len(bb.dec_blocks)


def _enc_call(blk, h, cos, sin, mask):
    _ENC_MASK[0] = mask
    return blk(h, cos, sin)


def _dec_call(blk, h, enc_out, cos, sin, xmask):
    _X_MASK[0] = xmask
    return blk(h, enc_out, cos, sin)


# ====================================================================== data
DOC_KEYS = ("document", "text", "judgment", "judgement", "doc", "source", "case_text", "content", "input")
SUM_KEYS = ("summary", "abstract", "headnote", "target", "reference", "output")


def _pick(r, keys):
    for k in keys:
        if k in r and r[k]:
            v = r[k]
            return " ".join(map(str, v)) if isinstance(v, list) else str(v)
    return None


def parse_summ(path):
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):
        if len(raw) == 1 and isinstance(next(iter(raw.values())), list):
            raw = next(iter(raw.values()))
        else:
            raw = [dict(v, id=k) if isinstance(v, dict) else {"id": k, "document": v} for k, v in raw.items()]
    out = []
    for i, r in enumerate(raw):
        did = str(r.get("id", r.get("doc_id", r.get("name", i))))
        out.append((did, _pick(r, DOC_KEYS) or "", _pick(r, SUM_KEYS)))
    return out, raw


def norm(t):
    return " ".join(str(t).split())


class Ex:
    __slots__ = ("id", "doc", "summ", "head", "tail", "sum_ids", "doc_words", "sum_words", "doc_tok", "key")


def tokenize(records, tok, head_tok, tail_tok):
    """head = first head_tok tokens of the document; tail = its last tail_tok tokens, kept only when the document is
    longer than head_tok + tail_tok (otherwise head holds the whole document and tail is None)."""
    exs = []
    for did, doc, summ in records:
        e = Ex()
        e.id, e.doc, e.summ = did, norm(doc), (norm(summ) if summ else None)
        e.doc_words, e.sum_words = len(e.doc.split()), (len(e.summ.split()) if e.summ else 0)
        e.key = hashlib.md5(e.doc[:3000].encode("utf-8", "ignore")).hexdigest()
        exs.append(e)
    budget = (head_tok + tail_tok) * 6
    dot = tok.encode(".").ids
    for i in range(0, len(exs), 1000):
        part = exs[i:i + 1000]
        short = [len(e.doc) <= budget for e in part]
        A = tok.encode_batch([e.doc if s else e.doc[:head_tok * 8] for e, s in zip(part, short)])
        Bt = tok.encode_batch(["" if s else e.doc[-max(1, tail_tok) * 8:] for e, s in zip(part, short)])
        C = tok.encode_batch([e.summ or "" for e in part])
        for e, s, a, b, c in zip(part, short, A, Bt, C):
            ids = a.ids or dot
            if s:
                e.doc_tok = len(ids)
                if len(ids) <= head_tok + tail_tok or tail_tok == 0:
                    e.head, e.tail = np.asarray(ids[:head_tok + tail_tok] if tail_tok else ids[:head_tok], np.int32), None
                else:
                    e.head, e.tail = np.asarray(ids[:head_tok], np.int32), np.asarray(ids[-tail_tok:], np.int32)
            else:
                e.doc_tok = -1                                       # longer than the token budget
                e.head = np.asarray(ids[:head_tok], np.int32)
                e.tail = np.asarray(b.ids[-tail_tok:], np.int32) if (tail_tok and b.ids) else None
            e.sum_ids = np.asarray(c.ids, np.int32)
    return exs


def dec_doc(e, doc_tok, mark, tail_tok=192):
    """Decoder document part: the whole document if it fits in doc_tok tokens, else its head + " [...]" + its tail."""
    tail = e.tail
    if tail is None:
        if len(e.head) <= doc_tok or tail_tok == 0:
            return [int(t) for t in e.head[:doc_tok]]
        tail = e.head[-tail_tok:]
    keep = doc_tok - len(mark) - len(tail)
    return [int(t) for t in e.head[:keep]] + list(mark) + [int(t) for t in tail]


def ed_chunks(e, k, chunk):
    h = e.head
    n = max(1, math.ceil(len(h) / chunk))
    pieces = [h[j * chunk:(j + 1) * chunk] for j in range(n)]
    if e.tail is None or k == 1:
        return pieces[:k]
    return pieces[:k - 1] + [e.tail]


def _target(e, sum_tok):
    s = [int(t) for t in e.sum_ids[:sum_tok]]
    return s + ([EOS_ID] if len(e.sum_ids) <= sum_tok else [])


def dec_train_batch(exs, cfg, sep, mark):
    xs, ys = [], []
    for e in exs:
        d = dec_doc(e, cfg["doc_tok"], mark, cfg.get("tail_tok", 192))
        tgt = _target(e, cfg["sum_tok"])
        seq = d + sep + tgt
        P = len(d) + len(sep)
        xs.append(seq[:-1])
        ys.append([-100] * (P - 1) + tgt)
    L = max(map(len, xs))
    x = torch.full((len(xs), L), PAD_ID, dtype=torch.long)
    y = torch.full((len(xs), L), -100, dtype=torch.long)
    for i, (a, b) in enumerate(zip(xs, ys)):
        x[i, :len(a)] = torch.tensor(a)
        y[i, :len(b)] = torch.tensor(b)
    return x, y


def ed_enc_batch(exs, k, chunk):
    per = [ed_chunks(e, k, chunk) for e in exs]
    B, K = len(exs), max(map(len, per))
    cmask = torch.zeros(B, K, dtype=torch.bool)
    rows = []
    for b, p in enumerate(per):
        for j, c in enumerate(p):
            cmask[b, j] = True
            rows.append(c if len(c) else np.asarray([PAD_ID], np.int32))
    T = max(map(len, rows))
    x = torch.full((len(rows), T), PAD_ID, dtype=torch.long)
    valid = torch.zeros((len(rows), T), dtype=torch.bool)
    for i, r in enumerate(rows):
        x[i, :len(r)] = torch.from_numpy(np.asarray(r, np.int64))
        valid[i, :len(r)] = True
    return x, valid, cmask


def ed_train_batch(exs, cfg):
    x, valid, cmask = ed_enc_batch(exs, cfg["k_enc"], cfg["chunk"])
    tg = [_target(e, cfg["sum_tok"]) for e in exs]
    L = max(map(len, tg))
    di = torch.full((len(exs), L), PAD_ID, dtype=torch.long)
    y = torch.full((len(exs), L), -100, dtype=torch.long)
    for i, t in enumerate(tg):
        di[i, 0] = BOS_ID
        if len(t) > 1:
            di[i, 1:len(t)] = torch.tensor(t[:-1])
        y[i, :len(t)] = torch.tensor(t)
    return x, valid, cmask, di, y


# ====================================================================== model wrapper
class SumModel(nn.Module):
    def __init__(self, bb, ckpt=True):
        super().__init__()
        self.bb, self.ckpt = bb, ckpt

    @property
    def kind(self):
        return self.bb.kind

    def _use_ckpt(self):
        return self.ckpt and torch.is_grad_enabled() and self.training

    def dec_hidden(self, x):
        bb, ck = self.bb, self._use_ckpt()
        h = bb.tok_emb(x)
        for blk in bb.blocks:
            h = checkpoint(blk, h, bb.rope_cos, bb.rope_sin, use_reentrant=False) if ck else blk(h, bb.rope_cos, bb.rope_sin)
        return bb.norm_f(h)

    def encode(self, x, valid, cmask):
        bb, ck = self.bb, self._use_ckpt()
        cos, sin = bb.rope_cos, bb.rope_sin
        try:
            h = bb.tok_emb(x)
            for blk in bb.enc_blocks:
                h = checkpoint(_enc_call, blk, h, cos, sin, valid, use_reentrant=False) if ck else _enc_call(blk, h, cos, sin, valid)
            h = bb.enc_norm(h)
        finally:
            _ENC_MASK[0] = None
        B, K = cmask.shape
        T = h.shape[1]
        full = h.new_zeros(B * K, T, h.shape[-1])
        full[cmask.reshape(-1)] = h
        m = valid.new_zeros(B * K, T)
        m[cmask.reshape(-1)] = valid
        return full.view(B, K * T, -1), m.view(B, K * T)

    def dec_states(self, dec_in, enc_out, xmask):
        bb, ck = self.bb, self._use_ckpt()
        try:
            h = bb.tok_emb(dec_in)
            for blk in bb.dec_blocks:
                h = checkpoint(_dec_call, blk, h, enc_out, bb.rope_cos, bb.rope_sin, xmask, use_reentrant=False) if ck \
                    else _dec_call(blk, h, enc_out, bb.rope_cos, bb.rope_sin, xmask)
            return bb.dec_norm(h)
        finally:
            _X_MASK[0] = None

    def target_logits(self, batch):
        """Logits only at the labelled (summary) positions -> (N, vocab) and labels (N,)."""
        if self.kind == "decoder":
            x, y = batch
            h = self.dec_hidden(x)
        else:
            x, valid, cmask, di, y = batch
            eo, xm = self.encode(x, valid, cmask)
            h = self.dec_states(di, eo, xm)
        sel = y != -100
        return self.bb.lm_head(h[sel]).float(), y[sel]


def make_batch(model, exs, cfg, sep, mark):
    return dec_train_batch(exs, cfg, sep, mark) if model.kind == "decoder" else ed_train_batch(exs, cfg)


# ====================================================================== AMP
def get_amp(device):
    if device == "cuda":
        dt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        return (lambda: torch.autocast(device_type="cuda", dtype=dt)), dt
    return (lambda: contextlib.nullcontext()), None


def make_scaler(device, amp_dtype):
    en = device == "cuda" and amp_dtype == torch.float16
    try:
        return torch.amp.GradScaler("cuda", enabled=en)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=en)


# ====================================================================== loss / training
def batch_loss(model, batch, amp, teacher=None, kd=None):
    with amp():
        lg, y = model.target_logits(batch)
    ce = F.cross_entropy(lg, y)
    if teacher is None:
        return ce, ce
    with torch.no_grad(), amp():
        tl, _ = teacher.target_logits(batch)
    T = kd["T"]
    kl = F.kl_div(F.log_softmax(lg / T, -1), F.softmax(tl.float() / T, -1), reduction="batchmean") * T * T
    return kd["alpha"] * ce + (1 - kd["alpha"]) * kl, ce


@torch.no_grad()
def eval_loss(model, exs, cfg, device, amp, sep, mark, bs=2):
    model.eval()
    tot, n = 0.0, 0
    for i in range(0, len(exs), bs):
        b = tuple(t.to(device) for t in make_batch(model, exs[i:i + bs], cfg, sep, mark))
        with amp():
            lg, y = model.target_logits(b)
        tot += F.cross_entropy(lg.float(), y, reduction="sum").item()
        n += y.numel()
    model.train()
    return tot / max(1, n)


def train(model, train_exs, dev_exs, cfg, device, amp, amp_dtype, sep, mark, teacher=None, kd=None, ada=None, log=print):
    rng = random.Random(cfg["seed"])
    named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    opt = torch.optim.AdamW([p for _, p in named], lr=cfg["lr"], betas=(0.9, 0.98), eps=1e-6, weight_decay=cfg["wd"])
    scaler = make_scaler(device, amp_dtype)
    dpe = min(len(train_exs), cfg["docs_per_epoch"])
    spe = math.ceil(dpe / (cfg["bs"] * cfg["accum"]))
    total = spe * cfg["epochs"]
    warm = max(1, int(cfg["warmup"] * min(total, cfg.get("expected_steps", total))))
    eval_every = max(1, cfg["eval_every"])
    keep = {n for n, _ in named} | {n for n, _ in model.named_buffers() if "rank_mask" in n}
    tot, trn = count_params(model)
    log(f"{len(train_exs)} train docs | {spe} steps/epoch x {cfg['epochs']} = {total} steps (time budget {cfg['max_min']} min) | "
        f"params {tot / 1e6:.1f}M, trainable {trn / 1e6:.3f}M ({100 * trn / tot:.2f}%)")
    hist, best, best_state, step, t0, ntok = [], float("inf"), None, 0, time.time(), 0
    model.train()
    if teacher is not None:
        teacher.eval()
    stop = False
    for ep in range(cfg["epochs"]):
        order = rng.sample(range(len(train_exs)), dpe)
        run, nrun = 0.0, 0
        for b0 in range(0, len(order), cfg["bs"]):
            exs = [train_exs[i] for i in order[b0:b0 + cfg["bs"]]]
            batch = tuple(t.to(device) for t in make_batch(model, exs, cfg, sep, mark))
            loss, ce = batch_loss(model, batch, amp, teacher, kd)
            if ada is not None:
                loss = loss + ada.reg_loss()
            if not torch.isfinite(loss):
                log("  non-finite loss - batch skipped")
                opt.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss / cfg["accum"]).backward()
            run += ce.item(); nrun += 1; ntok += int(batch[-1].ne(-100).sum())
            if (b0 // cfg["bs"] + 1) % cfg["accum"] == 0 or b0 + cfg["bs"] >= len(order):
                el = (time.time() - t0) / 60
                prog = max((step - warm) / max(1, total - warm), el / cfg["max_min"])
                mult = (step + 1) / warm if step < warm else max(0.03, 0.5 * (1 + math.cos(math.pi * min(1.0, prog))))
                for g in opt.param_groups:
                    g["lr"] = cfg["lr"] * mult
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for _, p in named], cfg["grad_clip"])
                if ada is not None:
                    ada.update(step, min(1.0, max(step / max(1, total), el / cfg["max_min"])))
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                step += 1
                if step % 10 == 0:
                    log(f"  ep {ep + 1} step {step}/{total} summary-token CE {run / max(1, nrun):.4f} | lr x{mult:.2f} | "
                        f"{ntok / max(1e-6, el * 60):,.0f} target tok/s | {el:.1f} min")
                stop = el > cfg["max_min"]
                if step % eval_every == 0 or stop or step == total:
                    dl = eval_loss(model, dev_exs, cfg, device, amp, sep, mark)
                    rec = dict(epoch=ep + 1, step=step, train_ce=run / max(1, nrun), dev_ce=dl, minutes=el)
                    if dl < best:
                        best = dl
                        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keep}
                        rec["best"] = True
                    hist.append(rec)
                    log("== " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))
                    run, nrun = 0.0, 0
                if stop:
                    log(f"time budget ({cfg['max_min']} min) reached - stopping")
                    break
        if stop:
            break
    if best_state is not None:
        model.load_state_dict(best_state, strict=False)
        log(f"restored best checkpoint (dev summary-token CE {best:.4f}, ppl {math.exp(best):.2f})")
    del opt
    return hist, best


# ====================================================================== generation
def _decode_loop(logits, caches, step_fn, mins, maxs, d, device, amp):
    B = logits.shape[0]
    n, rep, temp, top_p = d.get("no_repeat", 0), d.get("rep", 1.0), d.get("temperature", 0.0), d.get("top_p", 1.0)
    gen = [[] for _ in range(B)]
    done = [False] * B
    ngr = [dict() for _ in range(B)]
    gen_t = torch.zeros((B, 0), dtype=torch.long, device=device)
    ninf = -float("inf")
    for t in range(max(maxs) + 1):
        lg = logits.float()
        lg[:, PAD_ID] = ninf
        lg[:, BOS_ID] = ninf
        if rep != 1.0 and gen_t.shape[1] > 0:
            v = lg.gather(1, gen_t)
            lg.scatter_(1, gen_t, torch.where(v < 0, v * rep, v / rep))
        rows, cols = [], []
        for b in range(B):
            if done[b]:
                continue
            if t < mins[b]:
                rows.append(b); cols.append(EOS_ID)
            if n >= 2 and len(gen[b]) >= n - 1:
                ban = ngr[b].get(tuple(gen[b][len(gen[b]) - n + 1:]))
                if ban:
                    rows += [b] * len(ban); cols += list(ban)
        if rows:
            lg[torch.tensor(rows, device=device), torch.tensor(cols, device=device)] = ninf
        if temp > 0:
            lg = lg / temp
            if top_p < 1.0:
                sl, si = torch.sort(lg, descending=True)
                pr = sl.softmax(-1)
                sl = sl.masked_fill((pr.cumsum(-1) - pr) > top_p, ninf)
                lg = torch.full_like(lg, ninf).scatter(1, si, sl)
            nxt = torch.multinomial(lg.softmax(-1), 1).squeeze(1)
        else:
            nxt = lg.argmax(-1)
        if any(done):
            nxt = torch.where(torch.tensor(done, device=device), torch.full_like(nxt, EOS_ID), nxt)
        nl = nxt.tolist()
        for b in range(B):
            if done[b]:
                continue
            tk = nl[b]
            if tk == EOS_ID:
                done[b] = True
                continue
            if n >= 2 and len(gen[b]) >= n - 1:
                ngr[b].setdefault(tuple(gen[b][len(gen[b]) - n + 1:]), set()).add(tk)
            gen[b].append(tk)
            if len(gen[b]) >= maxs[b]:
                done[b] = True
        if all(done):
            break
        gen_t = torch.cat([gen_t, nxt[:, None]], 1)
        with amp():
            logits, caches = step_fn(nxt, caches)
    return gen


@torch.no_grad()
def generate(model, exs, lengths, dcfg, cfg, tok, device, amp, sep, mark, bs, log=None):
    """lengths: {id: target tokens}. Returns {id: summary text}. Decoder batches hold documents of equal input length
    (sorted by length, cut to the shortest in the batch - almost all long documents have exactly doc_tok tokens)."""
    model.eval()
    bb = model.bb
    out = {}
    lf, lo, hi = dcfg.get("lf", 1.0), cfg.get("min_frac", 0.8), cfg.get("max_frac", 1.25)
    if model.kind == "decoder":
        docs = {e.id: dec_doc(e, cfg["doc_tok"], mark, cfg.get("tail_tok", 192)) for e in exs}
        order = sorted(exs, key=lambda e: len(docs[e.id]))
    else:
        order = sorted(exs, key=lambda e: len(e.head) + (0 if e.tail is None else len(e.tail)))
    t0 = time.time()
    for s in range(0, len(order), bs):
        batch = order[s:s + bs]
        if model.kind == "decoder":
            L = min(len(docs[e.id]) for e in batch)
            x = torch.tensor([docs[e.id][:L] + list(sep) for e in batch], device=device)
            cap = max_positions(bb) - x.shape[1]
            with amp():
                logits, caches = bb.prefill(x)
            step_fn = bb.decode_step
        else:
            x, valid, cmask = ed_enc_batch(batch, cfg["k_enc"], cfg["chunk"])
            with amp():
                eo, xm = model.encode(x.to(device), valid.to(device), cmask.to(device))
            caches = [{"self": {}, "cross": {}} for _ in bb.dec_blocks]
            cap = max_positions(bb) - 1
            _X_MASK[0] = xm

            def step_fn(nxt, caches):
                h = bb.dec_hidden(nxt[:, None], None, caches)
                return bb.lm_head(h[:, -1]), caches

            with amp():
                bos = torch.full((len(batch), 1), BOS_ID, dtype=torch.long, device=device)
                logits = bb.lm_head(bb.dec_hidden(bos, eo, caches)[:, -1])
        cap = min(cap, cfg.get("gen_cap", 10 ** 9))
        maxs = [max(8, min(cap, int(round(lengths[e.id] * lf * hi)))) for e in batch]
        mins = [min(m - 1, int(round(lengths[e.id] * lf * lo))) for e, m in zip(batch, maxs)]
        try:
            ids = _decode_loop(logits, caches, step_fn, mins, maxs, dcfg, device, amp)
        finally:
            _X_MASK[0] = None
        for e, g in zip(batch, ids):
            t = norm(tok.decode(g))
            out[e.id] = t if t else " ".join(e.doc.split()[:80]) or "The court decided the case."
        if log is not None and (s // bs) % 5 == 0:
            log(f"    generated {min(s + bs, len(order))}/{len(order)} | {(time.time() - t0) / 60:.1f} min")
    model.train()
    return out


Writing sm_lib.py


In [4]:
%%writefile sm_peft.py
"""Adaptation methods for Phase 6, written from scratch (no `peft`, no `bitsandbytes`). apply(method, bb, cfg) -> info.
For the enc-dec every encoder AND decoder block is adapted (cross-attention projections included).
  lora     : W0 x + (alpha/r) B A x                                              (Hu et al., 2021)
  qlora    : the frozen base weights are stored in 4-bit NormalFloat (NF4, block size 64, one absmax per block,
             two codes packed per byte) and de-quantised on the fly; LoRA adapters on top in 16/32 bit (Dettmers et al., 2023)
  lorma    : (I + (alpha/r) B A) W0 x - multiplicative low-rank update           (Bihany, Patel, Modi, 2025) [compulsory]
  adalora  : P diag(lambda) Q, importance-based rank pruning + orthogonality regulariser (Zhang et al., 2023)
  houlsby  : bottleneck adapters after every attention (self + cross) and FFN sub-layer, layer norms trained (Houlsby et al., 2019)
  pfeiffer : one larger bottleneck adapter per block, after the FFN only          (Pfeiffer et al., 2021)
  kd       : student (every other layer of the teacher architecture), all weights trained on gold + teacher token distributions
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from sm_lib import blocks_all

DEFAULT_TARGETS = ("wq", "wk", "wv", "wo", "wg", "w_dkv", "w_ukv", "w_gate", "w_up", "w_down")


def replace_linears(bb, targets, factory):
    targets, n = set(targets), 0
    for blk in blocks_all(bb):
        for name, mod in list(blk.named_modules()):
            if name and isinstance(mod, nn.Linear) and name.split(".")[-1] in targets:
                parent_name, _, leaf = name.rpartition(".")
                parent = blk.get_submodule(parent_name) if parent_name else blk
                setattr(parent, leaf, factory(mod))
                n += 1
    return n


class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout=0.05):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


# ---------------------------------------------------------------- QLoRA: NF4 storage written from scratch
NF4_CODE = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
            -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
            0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
            0.7229568362236023, 1.0]


def nf4_quantize(W, block=64):
    code = torch.tensor(NF4_CODE, dtype=torch.float32)
    w = W.detach().float().reshape(-1)
    pad = (-w.numel()) % block
    if pad:
        w = F.pad(w, (0, pad))
    w = w.view(-1, block)
    absmax = w.abs().amax(1).clamp(min=1e-8)
    idx = torch.bucketize((w / absmax[:, None]).contiguous(), (code[1:] + code[:-1]) / 2).view(-1).to(torch.uint8)
    packed = (idx[0::2] << 4) | idx[1::2]
    return packed.contiguous(), absmax, code


def nf4_dequantize(packed, absmax, code, shape, block):
    idx = torch.stack([(packed >> 4).long(), (packed & 15).long()], 1).view(-1)
    w = (code[idx].view(-1, block) * absmax[:, None]).view(-1)
    n = 1
    for s in shape:
        n *= s
    return w[:n].view(*shape)


class QLoRALinear(nn.Module):
    def __init__(self, base, r, alpha, block=64, dropout=0.05):
        super().__init__()
        self.in_features, self.out_features = base.in_features, base.out_features
        self.shape, self.block, self.scale = tuple(base.weight.shape), block, alpha / r
        packed, absmax, code = nf4_quantize(base.weight, block)
        self.register_buffer("packed", packed)
        self.register_buffer("absmax", absmax)
        self.register_buffer("code", code)
        with torch.no_grad():
            W = base.weight.detach().float()
            self.rel_err = float((nf4_dequantize(packed, absmax, code, self.shape, block) - W).norm() / W.norm().clamp(min=1e-8))
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        W = nf4_dequantize(self.packed, self.absmax, self.code, self.shape, self.block).to(x.dtype)
        return F.linear(x, W) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        d = base.out_features
        self.base, self.r, self.scale = base, r, alpha / r
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.zeros(d, r))                 # B = 0 -> identity multiplier at init
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        h = self.base(x)
        return h + F.linear(F.linear(h, self.A), self.B) * self.scale


class AdaLoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.in_features, self.out_features = base.in_features, base.out_features
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.lam = nn.Parameter(torch.zeros(r))
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.register_buffer("rank_mask", torch.ones(r))

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(x, self.Q) * (self.lam * self.rank_mask), self.P) * self.scale

    def orth_loss(self):
        I = torch.eye(self.r, device=self.P.device)
        return ((self.P.t() @ self.P - I) ** 2).sum() + ((self.Q @ self.Q.t() - I) ** 2).sum()


class AdaLoRAController:
    """Cubic rank-budget schedule driven by training progress (0..1, steps or time budget, whichever is further), so the
    pruning finishes even when the run stops on its time budget. Importance = EMA(|w*grad|) x EMA(uncertainty)."""

    def __init__(self, model, init_r, target_r, t_init=0.1, t_final=0.8, delta_t=5, beta1=0.85, beta2=0.85, orth_reg=0.1):
        self.layers = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]
        self.ti, self.tf, self.dt = t_init, t_final, delta_t
        self.b0, self.bT = init_r * len(self.layers), target_r * len(self.layers)
        self.b1, self.b2, self.orth = beta1, beta2, orth_reg
        self.ipt = [None] * len(self.layers)
        self.unc = [None] * len(self.layers)
        self.frozen = False

    def reg_loss(self):
        return self.orth * sum(m.orth_loss() for m in self.layers) / len(self.layers)

    def budget(self, prog):
        if prog < self.ti:
            return self.b0
        if prog >= self.tf:
            return self.bT
        return int(self.bT + (self.b0 - self.bT) * (1 - (prog - self.ti) / (self.tf - self.ti)) ** 3)

    @torch.no_grad()
    def update(self, step, prog):
        scores = []
        for i, m in enumerate(self.layers):
            g = lambda p: p.grad if p.grad is not None else torch.zeros_like(p)
            s = torch.stack([(m.lam * g(m.lam)).abs(), (m.P * g(m.P)).abs().mean(0), (m.Q * g(m.Q)).abs().mean(1)]).float()
            s = torch.nan_to_num(s, nan=0.0, posinf=0.0, neginf=0.0)
            if self.ipt[i] is None:
                self.ipt[i], self.unc[i] = torch.zeros_like(s), torch.zeros_like(s)
            self.ipt[i] = self.b1 * self.ipt[i] + (1 - self.b1) * s
            self.unc[i] = self.b2 * self.unc[i] + (1 - self.b2) * (s - self.ipt[i]).abs()
            scores.append((self.ipt[i] * self.unc[i]).sum(0))
        if prog < self.ti or self.frozen or step % self.dt != 0:
            return
        b = self.budget(prog)
        allsc = torch.cat(scores)
        if b < allsc.numel():
            thr = torch.topk(allsc, b).values[-1]
            for m, sc in zip(self.layers, scores):
                m.rank_mask.copy_((sc >= thr).float())
        if prog >= self.tf:
            self.frozen = True                                       # final budget reached: keep the mask fixed

    def ranks(self):
        return [int(m.rank_mask.sum().item()) for m in self.layers]


class Adapter(nn.Module):
    """Wraps a sub-layer: out = f(x); out + up(gelu(down(out))). Up-projection starts at 0 (identity at init)."""

    def __init__(self, inner, d, m):
        super().__init__()
        self.inner = inner
        self.down, self.up = nn.Linear(d, m), nn.Linear(m, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, *a, **k):
        h = self.inner(*a, **k)
        return h + self.up(F.gelu(self.down(h)))


def apply(method, bb, cfg):
    """The caller has already set requires_grad=False on every backbone parameter (and patched the enc-dec masks)."""
    info = {"method": method}
    if method in ("full", "kd"):
        for p in bb.parameters():
            p.requires_grad = True
        n = sum(1 for _ in bb.parameters())
    elif method == "lora":
        n = replace_linears(bb, cfg.get("targets", DEFAULT_TARGETS), lambda l: LoRALinear(l, cfg["r"], cfg["alpha"]))
    elif method == "qlora":
        before = sum(p.numel() * p.element_size() for p in bb.parameters())
        n = replace_linears(bb, cfg.get("targets", DEFAULT_TARGETS), lambda l: QLoRALinear(l, cfg["r"], cfg["alpha"], cfg.get("block", 64)))
        q = [m for m in bb.modules() if isinstance(m, QLoRALinear)]
        info["nf4_rel_error"] = float(sum(m.rel_err for m in q) / max(1, len(q)))
        frozen_lin = sum(m.packed.numel() + m.absmax.numel() * 4 for m in q)
        info["base_linear_bytes_nf4"] = int(frozen_lin)
        info["backbone_bytes_fp32_before"] = int(before)
        print(f"[qlora] {len(q)} linears stored in NF4: {frozen_lin / 1e6:.1f} MB (fp32 backbone was {before / 1e6:.1f} MB) | "
              f"mean relative weight error {info['nf4_rel_error']:.4f}")
    elif method == "lorma":
        n = replace_linears(bb, cfg.get("targets", DEFAULT_TARGETS), lambda l: LoRMALinear(l, cfg["r"], cfg["alpha"]))
    elif method == "adalora":
        n = replace_linears(bb, cfg.get("targets", DEFAULT_TARGETS), lambda l: AdaLoRALinear(l, cfg["init_r"], cfg["alpha"]))
    elif method in ("houlsby", "pfeiffer"):
        d, m = bb.cfg.d_model, cfg["bottleneck"]
        attrs = ("attn", "self_attn", "cross_attn", "ffn") if method == "houlsby" else ("ffn",)
        n = 0
        for blk in blocks_all(bb):
            for a in attrs:
                if hasattr(blk, a):
                    setattr(blk, a, Adapter(getattr(blk, a), d, m))
                    n += 1
        if method == "houlsby":                                     # Houlsby: layer norms are trained as well
            for blk in blocks_all(bb):
                for mod in blk.modules():
                    if type(mod).__name__ in ("RMSNorm", "LayerNorm"):
                        for p in mod.parameters():
                            p.requires_grad = True
    else:
        raise ValueError(f"unknown method {method}")
    assert n > 0, f"{method}: nothing to adapt"
    info["n_modules"] = n
    print(f"[{method}] applied ({n} modules)", {k: v for k, v in cfg.items() if k != "targets"})
    return info


Writing sm_peft.py


In [5]:
%%writefile sm_run.py
"""Phase 6 summarisation - one experiment. Usage: python sm_run.py <method> <arch> [tag]
method: lora | qlora | lorma | adalora | houlsby | pfeiffer | kd | full      arch: gqa | mla | retnet | encdec
Env: SM_SMOKE=1 (tiny pipeline check), SM_OVERRIDES='{"K_ENC": 1}' (override any upper-case setting below)."""
import os, sys, json, math, time, random, glob, hashlib
from pathlib import Path
import numpy as np
import torch

METHOD, ARCH = sys.argv[1], sys.argv[2]
TAG = sys.argv[3] if len(sys.argv) > 3 else ""
HF_TOKEN = os.environ.get("HF_TOKEN") or None
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SEED = 42

# ---- input / output lengths
DOC_TOK, TAIL_TOK = 1152, 192      # decoders: document part of the 2048 window = first 960 tokens + " [...]" + last 192
SUM_TOK_DEC = 880                  # decoders: summary tokens trained on (1152 + 6 + 880 + 1 <= 2048)
CHUNK, K_ENC = 1024, 4             # enc-dec: up to 4 encoder chunks of 1024 (first 3 + the final chunk) = 4096 doc tokens
SUM_TOK_ENC = 1000                 # enc-dec: decoder positions are limited to 1024 by the Phase-1 RoPE table
# ---- optimisation
EPOCHS, DOCS_PER_EPOCH = 4, 100000
BS, ACCUM = 2, 8
WD, WARMUP, GRAD_CLIP = 0.0, 0.05, 1.0
MAX_MIN = 60                       # training time budget (minutes); best dev-loss checkpoint is kept
EVAL_EVERY = 60                    # optimiser steps between dev-loss checks
# ---- evaluation / generation
DEV_MAX = 300                      # held-out documents (Phase 6 has no dev split): ~5% of train, fixed by id hash
DEV_LOSS_N, SWEEP_N, DEV_GEN_N = 64, 24, 120
GEN_BS = 16
DECODES = {                        # compared on SWEEP_N dev documents, the best (BERTScore, else ROUGE-1) is used
    "greedy_nr3": dict(no_repeat=3, rep=1.0, lf=1.0),
    "greedy_nr4_rep1.15": dict(no_repeat=4, rep=1.15, lf=1.0),
    "greedy_nr3_len0.7": dict(no_repeat=3, rep=1.0, lf=0.7),
}
MIN_FRAC, MAX_FRAC = 0.8, 1.25     # generation length window around the length model's prediction

MCFG_ALL = {
    "lora":     dict(lr=3e-4, r=32, alpha=64),
    "qlora":    dict(lr=3e-4, r=32, alpha=64, block=64),
    "lorma":    dict(lr=3e-4, r=16, alpha=16),
    "adalora":  dict(lr=5e-4, init_r=16, target_r=8, alpha=16, orth_reg=0.1, delta_t=5),
    "houlsby":  dict(lr=5e-4, bottleneck=128),
    "pfeiffer": dict(lr=5e-4, bottleneck=256),
    "kd":       dict(lr=5e-5, keep=None, alpha=0.5, T=2.0, teacher="sm_lora_gqa"),
    "full":     dict(lr=3e-5),
}
for _k, _v in json.loads(os.environ.get("SM_OVERRIDES", "{}")).items():
    globals()[_k] = _v
SMOKE = os.environ.get("SM_SMOKE") == "1"
if SMOKE:
    EPOCHS, DOCS_PER_EPOCH, MAX_MIN, EVAL_EVERY = 1, 16, 3, 4
    DEV_LOSS_N, SWEEP_N, DEV_GEN_N, ACCUM = 4, 4, 6, 2
MCFG = dict(MCFG_ALL[METHOD])

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
WORK = ROOT / f"sm_{METHOD}_{ARCH}{TAG}{'_smoke' if SMOKE else ''}"
WORK.mkdir(parents=True, exist_ok=True)
print("device:", device, "| out:", WORK, "| cfg:", MCFG, flush=True)

import sm_lib, sm_peft
from sm_metrics import evaluate_texts, fmt, LengthModel, write_prediction, ROUGE_IMPL, redundancy
from huggingface_hub import hf_hub_download, HfApi
from tokenizers import Tokenizer
amp, AMP_DTYPE = sm_lib.get_amp(device)
print("ROUGE implementation:", ROUGE_IMPL)

# ------------------------------------------------------------------ data
files = HfApi().list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
p6 = [f for f in files if "Phase_6" in f or "Summ" in f]
print("Phase-6 files:", p6)


def fetch(split):
    m = [f for f in p6 if f.endswith("/" + split + ".json") or f.endswith(split + ".json")]
    assert m, f"{split}.json not found in {p6}"
    return hf_hub_download(HF_DATASET_REPO, m[0], repo_type="dataset", token=HF_TOKEN)


recs = {}
for part in ("train", "test"):
    recs[part], raw = sm_lib.parse_summ(fetch(part))
    r0 = recs[part][0]
    print(f"{part}: {len(recs[part])} docs | raw {type(raw).__name__} | keys "
          f"{list(raw[0].keys()) if isinstance(raw, list) and raw and isinstance(raw[0], dict) else '-'} | example id={r0[0]!r} "
          f"doc={r0[1][:150]!r} | summary={(r0[2] or '')[:150]!r}", flush=True)
seen, tr = set(), []
for r in recs["train"]:
    if r[0] not in seen and r[1].strip() and r[2] and r[2].strip():
        seen.add(r[0]); tr.append(r)
print(f"train after id de-duplication / dropping empty: {len(tr)} (removed {len(recs['train']) - len(tr)})")
# fixed held-out split (same for every run): the documents with the smallest md5(id)
hid = sorted(tr, key=lambda r: hashlib.md5(r[0].encode()).hexdigest())
n_dev = min(DEV_MAX, max(30, int(0.05 * len(tr))))
recs["dev"], recs["train"] = hid[:n_dev], hid[n_dev:]
test_ids = [r[0] for r in recs["test"]]
assert len(set(test_ids)) == len(test_ids), "duplicate test ids"
if SMOKE:
    recs["train"], recs["test"] = recs["train"][:40], recs["test"][:6]

export_dir = sm_lib.find_p1_export(ARCH)
assert export_dir is not None, f"Phase-1 export for {ARCH} not found under /kaggle/input"
tok = Tokenizer.from_file(str(export_dir / "tokenizer.json"))
SEP = tok.encode(sm_lib.SEP_TEXT).ids
MARK = tok.encode(sm_lib.MARK_TEXT).ids
if ARCH == "encdec":
    HEAD, TAIL = (K_ENC - 1) * CHUNK if K_ENC > 1 else CHUNK, (CHUNK if K_ENC > 1 else 0)
else:
    HEAD, TAIL = DOC_TOK, TAIL_TOK
t0 = time.time()
ex = {p: sm_lib.tokenize(recs[p], tok, HEAD, TAIL) for p in recs}
print(f"tokenised in {time.time() - t0:.0f}s | input: {'enc-dec ' + str(K_ENC) + ' x ' + str(CHUNK) if ARCH == 'encdec' else 'decoder ' + str(DOC_TOK) + ' doc tokens (tail ' + str(TAIL_TOK) + ')'}", flush=True)
train_ex, dev_ex, test_ex = ex["train"], ex["dev"], ex["test"]
SUM_TOK = SUM_TOK_ENC if ARCH == "encdec" else SUM_TOK_DEC
for p in ("train", "dev", "test"):
    dw = np.array([e.doc_words for e in ex[p]])
    msg = f"{p}: {len(ex[p])} docs | doc words mean {dw.mean():.0f} median {np.median(dw):.0f} p90 {np.percentile(dw, 90):.0f} max {dw.max()}"
    msg += f" | docs longer than the input budget {np.mean([e.tail is not None or e.doc_tok < 0 for e in ex[p]]):.2f}"
    if p != "test":
        sw = np.array([e.sum_words for e in ex[p]]); st = np.array([len(e.sum_ids) for e in ex[p]])
        msg += (f" | summary words mean {sw.mean():.0f} median {np.median(sw):.0f} p10 {np.percentile(sw, 10):.0f} p90 {np.percentile(sw, 90):.0f}"
                f" | summary tokens median {np.median(st):.0f}, > {SUM_TOK}: {np.mean(st > SUM_TOK):.2f} | compression {np.median(sw / np.maximum(dw, 1)):.3f}")
    print(msg)
# how abstractive are the references? share of summary words that never occur in the document
nov = []
for e in train_ex[:400]:
    dws = set(e.doc.lower().split())
    sws = e.summ.lower().split()
    nov.append(np.mean([w not in dws for w in sws]) if sws else 0)
print(f"novel summary words (not in the document): {100 * np.mean(nov):.1f}% | summary repeated 4-grams "
      f"{100 * np.mean([redundancy(e.summ) for e in train_ex[:400]]):.1f}%")
tkeys = {e.key for e in train_ex}
print(f"overlap with train (same text head): dev {sum(e.key in tkeys for e in dev_ex)} | test {sum(e.key in tkeys for e in test_ex)}")
LM = LengthModel([e.doc_words for e in train_ex], [e.sum_words for e in train_ex], [len(e.sum_ids) for e in train_ex])
print("length model:", LM.describe())
LENS = {e.id: LM.tokens(e.doc_words) for e in dev_ex + test_ex}

# ------------------------------------------------------------------ model
arch_mod, bcfg, bb = sm_lib.load_p1(export_dir)
P1_PARAMS = sum(p.numel() for p in bb.parameters())
print(f"Phase-1 {ARCH}: {P1_PARAMS / 1e6:.1f}M params, kind {bb.kind}, RoPE positions {sm_lib.max_positions(bb)}")
if bb.kind == "decoder":
    assert DOC_TOK + len(SEP) + SUM_TOK + 1 <= sm_lib.max_positions(bb) + 1
else:
    assert SUM_TOK + 1 <= sm_lib.max_positions(bb) and CHUNK <= sm_lib.max_positions(bb)

TEACHER, TEACHER_INFO = None, None
if METHOD == "kd":
    tdir = ROOT / (MCFG["teacher"] + ("_smoke" if SMOKE else ""))
    cands = [tdir] + [Path(p).parent for p in glob.glob(f"/kaggle/input/**/{MCFG['teacher']}/adapter.pt", recursive=True)]
    tdir = next((c for c in cands if (c / "adapter.pt").exists()), None)
    assert tdir is not None, "KD needs the teacher run's adapter.pt (run lora gqa first)"
    tres = json.load(open(tdir / "results.json"))
    assert tres["arch"] == ARCH, "teacher and student must share the tokenizer / architecture family"
    _, _, tbb = sm_lib.load_p1(export_dir)
    for p in tbb.parameters():
        p.requires_grad = False
    sm_peft.apply(tres["method"], tbb, tres["mcfg"])
    tbb.load_state_dict(torch.load(tdir / "adapter.pt", map_location="cpu"), strict=False)
    for p in tbb.parameters():
        p.requires_grad = False
    TEACHER = sm_lib.SumModel(tbb, ckpt=False).to(device).eval()
    TEACHER_INFO = dict(run=tres["run"], dev=tres.get("dev"), params_total=tres.get("params_total"))
    nL = len(bb.blocks)
    keep = MCFG["keep"] or list(range(0, nL, 2))
    bb, bcfg = sm_lib.make_student_bb(arch_mod, bcfg, bb, keep)
    print(f"teacher {tres['run']} | student keeps layers {keep} of {nL}: {sum(p.numel() for p in bb.parameters()) / 1e6:.1f}M params")

sm_lib.patch_encdec(bb)                        # key-padding masks; must happen before adapters wrap the attention modules
for p in bb.parameters():
    p.requires_grad = False
INFO = sm_peft.apply(METHOD, bb, MCFG)
model = sm_lib.SumModel(bb, ckpt=True).to(device)
tot, trn = sm_lib.count_params(model)
print(f"model {tot / 1e6:.1f}M params | trainable {trn / 1e6:.3f}M", flush=True)

TCFG = dict(lr=MCFG["lr"], wd=WD, warmup=WARMUP, epochs=EPOCHS, docs_per_epoch=DOCS_PER_EPOCH, bs=BS, accum=ACCUM,
            grad_clip=GRAD_CLIP, max_min=MAX_MIN, seed=SEED, eval_every=EVAL_EVERY, doc_tok=DOC_TOK, tail_tok=TAIL_TOK, sum_tok=SUM_TOK,
            k_enc=K_ENC, chunk=CHUNK, min_frac=MIN_FRAC, max_frac=MAX_FRAC, expected_steps=400,
            gen_cap=48 if SMOKE else 10 ** 9)
ADA = sm_peft.AdaLoRAController(model, MCFG["init_r"], MCFG["target_r"], delta_t=MCFG["delta_t"], orth_reg=MCFG["orth_reg"]) \
    if METHOD == "adalora" else None
KD = dict(alpha=MCFG["alpha"], T=MCFG["T"]) if METHOD == "kd" else None
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
t0 = time.time()
hist, best = sm_lib.train(model, train_ex, dev_ex[:DEV_LOSS_N], TCFG, device, amp, AMP_DTYPE, SEP, MARK, teacher=TEACHER, kd=KD, ada=ADA)
TRAIN_MIN = (time.time() - t0) / 60
PEAK = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0
if ADA is not None:
    r = ADA.ranks()
    print(f"AdaLoRA final ranks: mean {np.mean(r):.1f} | min {min(r)} max {max(r)} | {r}")
del TEACHER
if device == "cuda":
    torch.cuda.empty_cache()
keep_names = {n for n, p in model.bb.named_parameters() if p.requires_grad} | {n for n, _ in model.bb.named_buffers() if "rank_mask" in n}
if METHOD != "kd":                                  # adapter-only weights (small); the KD teacher is rebuilt from these
    torch.save({k: v.detach().cpu() for k, v in model.bb.state_dict().items() if k in keep_names}, WORK / "adapter.pt")

# ------------------------------------------------------------------ decoding choice, dev, test


def gen(exs, dcfg, log=None):
    return sm_lib.generate(model, exs, LENS, dcfg, TCFG, tok, device, amp, SEP, MARK, GEN_BS if ARCH != "encdec" else 8, log=log)


sweep_ex, sweep = dev_ex[:SWEEP_N], {}
for name, dc in DECODES.items():
    t1 = time.time()
    out = gen(sweep_ex, dc)
    sweep[name] = evaluate_texts([out[e.id] for e in sweep_ex], [e.summ for e in sweep_ex], device)
    print(f"decode {name:>20}: {fmt(sweep[name])} | {time.time() - t1:.0f}s", flush=True)
# BERTScore (roberta-large) reads only the first ~510 sub-word tokens, so decodings that differ only in length after
# ~400 words tie on it: compare BERTScore at 3 decimals, break ties with ROUGE-1 (which does see the full length)
BEST_DEC = max(sweep, key=lambda k: (round(sweep[k]["score"], 3), sweep[k]["rouge1"]))
print("chosen decoding:", BEST_DEC, DECODES[BEST_DEC])
gex = dev_ex[:DEV_GEN_N]
t1 = time.time()
dev_out = gen(gex, DECODES[BEST_DEC])
DEV = evaluate_texts([dev_out[e.id] for e in gex], [e.summ for e in gex], device)
DEV_SPD = len(gex) / max(1e-6, time.time() - t1)
LEAD = evaluate_texts([" ".join(e.doc.split()[:int(LM.words(e.doc_words))]) for e in gex], [e.summ for e in gex], device)
print(f"DEV ({len(gex)} docs): {fmt(DEV)}")
print(f"reference point, not a model: first-N-words of the document (N from the length model): {fmt(LEAD)}")
e0 = gex[0]
print(f"\nEXAMPLE {e0.id}\n  REFERENCE: {e0.summ[:700]}\n  GENERATED: {dev_out[e0.id][:700]}\n", flush=True)
json.dump({e.id: e.summ for e in gex}, open(WORK / "dev_truth.json", "w"), ensure_ascii=False)
write_prediction([e.id for e in gex], dev_out, WORK / "dev_prediction.json", WORK / "dev_submission.zip")
json.dump({"run": WORK.name, "summaries": dev_out}, open(WORK / "dev_pred.json", "w"), ensure_ascii=False)
json.dump({e.id: " ".join(e.doc.split()[:int(LM.words(e.doc_words))]) for e in gex}, open(WORK / "dev_lead.json", "w"), ensure_ascii=False)

t1 = time.time()
test_out = gen(test_ex, DECODES[BEST_DEC], log=print)
TEST_SPD = len(test_ex) / max(1e-6, time.time() - t1)
ids = [e.id for e in test_ex]
write_prediction(ids, test_out, WORK / "prediction.json", WORK / "submission.zip")
json.dump({"run": WORK.name, "summaries": test_out}, open(WORK / "test_pred.json", "w"), ensure_ascii=False)
tw = [len(test_out[i].split()) for i in ids]
print(f"test: {len(ids)} summaries | words min {min(tw)} median {int(np.median(tw))} max {max(tw)} | "
      f"repeated 4-grams {100 * np.mean([redundancy(test_out[i]) for i in ids]):.1f}% | {TEST_SPD * 60:.1f} docs/min"
      + (" | SMOKE: only the first test documents - not a valid submission" if SMOKE else ""))

res = dict(run=WORK.name, method=METHOD, arch=ARCH, mcfg=MCFG, info=INFO, params_total=tot, params_trainable=trn, phase1_params=P1_PARAMS,
           train_min=TRAIN_MIN, peak_gpu_gb=PEAK, best_dev_ce=best, history=hist, decode_sweep=sweep, decode=BEST_DEC, dev=DEV,
           lead_baseline=LEAD, dev_docs_per_min=DEV_SPD * 60, test_docs_per_min=TEST_SPD * 60, k_enc=K_ENC if ARCH == "encdec" else None,
           doc_tok=DOC_TOK if ARCH != "encdec" else K_ENC * CHUNK, sum_tok=SUM_TOK, length_model=LM.describe(), teacher=TEACHER_INFO,
           n_train=len(train_ex), n_dev=len(dev_ex), n_test=len(test_ex))
json.dump(res, open(WORK / "results.json", "w"), indent=1, default=float)
bs_txt = "n/a" if DEV["bertscore"] is None else f"{DEV['bertscore']:.4f}"
print(f"\nSUMMARY {WORK.name}: dev BERTScore {bs_txt} | R1 {DEV['rouge1']:.4f} | RL {DEV['rougeL']:.4f} | dev CE {best:.3f} | "
      f"params {tot / 1e6:.1f}M trainable {trn / 1e6:.2f}M | train {TRAIN_MIN:.0f} min | peak {PEAK:.1f} GB | decode {BEST_DEC}")
if METHOD == "kd" and TEACHER_INFO and TEACHER_INFO.get("dev"):
    td = TEACHER_INFO["dev"]
    print(f"KD: student {DEV['score']:.4f} vs teacher {td['score']:.4f} ({100 * DEV['score'] / max(1e-9, td['score']):.1f}%) with "
          f"{100 * tot / max(1, TEACHER_INFO['params_total']):.0f}% of the parameters")
print("RUN_DONE", WORK.name, flush=True)


Writing sm_run.py


In [6]:
%%writefile sm_ens.py
"""Phase 6 - ensemble of finished runs by consensus (MBR) selection: for every document, among the runs' summaries pick
the one with the highest mean ROUGE-1 F1 against the other runs' summaries (no reference needed). The member set is
chosen greedily on the held-out documents (BERTScore when available, else ROUGE-1).
Writes submissions/ensemble_mbr_topk.zip and submissions/ensemble_mbr_all.zip. Usage: python sm_ens.py [--smoke]"""
import json, glob, sys
from pathlib import Path
sys.path.insert(0, ".")
from sm_metrics import evaluate_texts, fmt, mbr_select, write_prediction

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
SUB = ROOT / "submissions"; SUB.mkdir(exist_ok=True)
SMOKE = "--smoke" in sys.argv
device = None
try:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    pass

dirs = sorted({str(Path(p).parent) for p in glob.glob(str(ROOT / "sm_*/test_pred.json"))
               if ("_smoke" in p) == SMOKE and Path(p).with_name("dev_pred.json").exists()})
runs = {}
for d in dirs:
    jd, jt = json.load(open(Path(d) / "dev_pred.json")), json.load(open(Path(d) / "test_pred.json"))
    runs[jd["run"]] = (jd["summaries"], jt["summaries"])
names = sorted(runs)
assert names, "no finished runs found"
truth = json.load(open(Path(dirs[0]) / "dev_truth.json"))
dev_ids = [i for i in truth if all(i in runs[n][0] for n in names)]
test_ids = sorted(set.intersection(*[set(runs[n][1]) for n in names]))
refs = [truth[i] for i in dev_ids]
print(f"{len(names)} runs | dev docs {len(dev_ids)} | test docs {len(test_ids)}")

single = {}
for n in names:
    single[n] = evaluate_texts([runs[n][0][i] for i in dev_ids], refs, device)
    print(f"{n:>30}  {fmt(single[n])}")


def mbr(members, split):
    k = 0 if split == "dev" else 1
    ids = dev_ids if split == "dev" else test_ids
    out, pick = {}, {}
    for i in ids:
        c = [runs[m][k][i] for m in members]
        j = mbr_select(c)
        out[i], pick[members[j]] = c[j], pick.get(members[j], 0) + 1
    return out, pick


# with two candidates the consensus score is symmetric (MBR just returns the first), so member sets start at 3:
# rank runs by their own dev score and keep the top-k (k = 3..all) that scores best on dev
ranked = sorted(names, key=lambda n: -single[n]["score"])
members, best = ranked[:1], single[ranked[0]]["score"]
for k in range(3, len(ranked) + 1):
    o, _ = mbr(ranked[:k], "dev")
    m = evaluate_texts([o[i] for i in dev_ids], refs, device)
    print(f"MBR over top-{k} runs: {fmt(m)}")
    if m["score"] > best + 1e-4:
        members, best = ranked[:k], m["score"]
print(f"best member set on dev: {members} ({best:.4f})")

out = {}
for name, mem in (("ensemble_mbr_topk", members), ("ensemble_mbr_all", [n for n in names if "_1chunk" not in n])):
    if len(mem) < 2:
        print(name, "skipped (fewer than 2 members)")
        continue
    o_dev, pick_dev = mbr(mem, "dev")
    m = evaluate_texts([o_dev[i] for i in dev_ids], refs, device)
    o_test, pick_test = mbr(mem, "test")
    zname = ("smoke_" if SMOKE else "") + name
    write_prediction(test_ids, o_test, ROOT / f"prediction_{zname}.json", SUB / f"{zname}.zip")
    print(f"{zname.upper()} members {mem}\n   dev: {fmt(m)}\n   chosen per run on test: {pick_test}")
    out[name] = dict(members=mem, dev=m, picks_dev=pick_dev, picks_test=pick_test)
out["single"] = single
json.dump(out, open(ROOT / "ensemble_results.json", "w"), indent=1, default=float)
print("ENS_DONE")


Writing sm_ens.py


In [7]:
%%writefile evaluate_phase6.py
#!/usr/bin/env python3
"""Local evaluator for Phase 6.

Requires: rouge-score, bert-score, torch.
Truth JSON: {"doc_id":"reference summary", ...}
Prediction JSON: either {"doc_id":"summary", ...} or [{"id":"...","summary":"..."}, ...]
"""
import argparse,json,sys
try:
    from rouge_score import tokenizers, rouge_scorer
    import bert_score
except ImportError as e:
    print('Missing dependency. Install requirements.txt first.', file=sys.stderr); raise

def load(path):
    with open(path,encoding='utf-8') as f: return json.load(f)

def lcs(a,b):
    if len(a)>len(b): a,b=b,a
    masks={}
    for i,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<i)
    s=0
    for t in b:
        m=masks.get(t,0); x=m|s; y=(s<<1)|1; s=x & ~(x-y)
    return s.bit_count() if hasattr(int,'bit_count') else bin(s).count('1')

def read_pred(x):
    if isinstance(x,dict): out=x
    elif isinstance(x,list):
        out={}
        for r in x:
            if not isinstance(r,dict) or 'id' not in r or 'summary' not in r: raise ValueError('Each prediction item must contain id and summary.')
            if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
            out[r['id']]=r['summary']
    else: raise ValueError('Prediction must be a JSON object or list of objects.')
    for k,v in out.items():
        if not isinstance(k,str) or not isinstance(v,str) or not v.strip(): raise ValueError(f'Invalid summary for ID {k!r}.')
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 6 summaries on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); a=ap.parse_args()
    t=load(a.truth); p=read_pred(load(a.prediction))
    if not isinstance(t,dict) or not t: raise ValueError('Truth must be a non-empty JSON object.')
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    preds=[p[k].strip() for k in t]; refs=[t[k].strip() for k in t]
    scorer=rouge_scorer.RougeScorer(['rouge1'],use_stemmer=True); tok=tokenizers.DefaultTokenizer(use_stemmer=True)
    r1=rl=0.0
    for pr,rf in zip(preds,refs):
        s=scorer.score(rf,pr); r1+=s['rouge1'].fmeasure
        pt=tok.tokenize(pr); rt=tok.tokenize(rf)
        if not pt and not rt: x=1.0
        elif not pt or not rt: x=0.0
        else:
            z=lcs(pt,rt); pp=z/len(pt); rr=z/len(rt); x=0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)
        rl+=x
    n=len(t); r1/=n; rl/=n
    try:
        import torch; device='cuda' if torch.cuda.is_available() else 'cpu'
    except ImportError: device=None
    _,_,f=bert_score.score(preds,refs,lang='en',batch_size=8,verbose=False,**({'device':device} if device else {}))
    print(f'Documents: {n}'); print(f'BERTScore F1: {float(f.mean().item()):.6f}'); print(f'ROUGE-1: {r1:.6f}'); print(f'ROUGE-L: {rl:.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)


Writing evaluate_phase6.py


In [8]:
# ===== Phase 6 driver: every experiment is its own process, one queue per T4; then KD (needs the LoRA-GQA teacher) +
# a long-document ablation; then the MBR ensemble and the official evaluator on every run's held-out predictions.
# Set SMOKE = True for a ~20-minute pipeline check first.
import os, sys, subprocess, time, glob, shutil, json, py_compile
from pathlib import Path
import torch
SMOKE = False       # True = tiny pipeline check (passed on 29 Oct). False = the real runs (~6.5 h on 2 x T4).
os.system(f"{sys.executable} -m pip -q install rouge-score bert-score")
for f in ("sm_metrics.py", "sm_lib.py", "sm_peft.py", "sm_run.py", "sm_ens.py", "evaluate_phase6.py"):
    py_compile.compile(f, doraise=True)
os.environ["HF_TOKEN"] = HF_TOKEN or ""
# download roberta-large (the BERTScore model) once, before the parallel runs start
r = subprocess.run([sys.executable, "-c", "from bert_score import BERTScorer; s = BERTScorer(lang='en', device='cpu'); "
                    "print('BERTScore ready', float(s.score(['the appeal is dismissed'], ['the appeal was dismissed'])[2][0]))"],
                   capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-1500:] if r.returncode else "")
NGPU = max(1, torch.cuda.device_count())
# (method, arch, tag, overrides)
QUEUES = {0: [("lora", "gqa", "", {}), ("qlora", "retnet", "", {}), ("pfeiffer", "mla", "", {})],
          1: [("lorma", "encdec", "", {}), ("houlsby", "encdec", "", {}), ("adalora", "mla", "", {})]}
FINAL = {0: [("kd", "gqa", "", {})],                                  # student = 8-layer GQA, teacher = the LoRA-GQA run
         1: [("lorma", "encdec", "_1chunk", {"K_ENC": 1})]}          # ablation: one 1024-token encoder window
if NGPU == 1:
    QUEUES = {0: QUEUES[0] + QUEUES[1]}
    FINAL = {0: FINAL[0] + FINAL[1]}
LOG = Path("/kaggle/working/logs"); LOG.mkdir(parents=True, exist_ok=True)
T0 = time.time()

def tail(path, n=1):
    try:
        return open(path).read().strip().splitlines()[-n:]
    except Exception:
        return []

def run_queues(queues):
    pend = {g: list(q) for g, q in queues.items()}
    cur, status, last = {g: None for g in queues}, {}, 0
    while True:
        for g in queues:
            if cur[g] is not None and cur[g][0].poll() is not None:
                p, f, name = cur[g]; f.close(); status[name] = p.returncode
                print(f"[{(time.time() - T0) / 60:6.1f} min] GPU{g} finished {name} rc={p.returncode}\n    " + "\n    ".join(tail(LOG / f"{name}.log", 4)), flush=True)
                cur[g] = None
            if cur[g] is None and pend[g]:
                m, a, tag, ov = pend[g].pop(0); name = f"{m}_{a}{tag}"
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), SM_SMOKE="1" if SMOKE else "0", PYTHONUNBUFFERED="1",
                           SM_OVERRIDES=json.dumps(ov), TOKENIZERS_PARALLELISM="false")
                f = open(LOG / f"{name}.log", "w")
                cur[g] = (subprocess.Popen([sys.executable, "sm_run.py", m, a, tag], stdout=f, stderr=subprocess.STDOUT, env=env), f, name)
                print(f"[{(time.time() - T0) / 60:6.1f} min] GPU{g} started {name}", flush=True)
        if all(c is None for c in cur.values()):
            return status
        if time.time() - last > 300:
            last = time.time()
            for g, c in cur.items():
                if c is not None:
                    print(f"[{(time.time() - T0) / 60:6.1f} min]   GPU{g} {c[2]}: {tail(LOG / (c[2] + '.log'))}", flush=True)
        time.sleep(15)

print("GPUs:", NGPU, "| queues:", QUEUES, "| final:", FINAL, "| smoke:", SMOKE, flush=True)
status = run_queues(QUEUES)
status.update(run_queues(FINAL))
print("\nreturn codes:", status)
sub = Path("/kaggle/working/submissions"); sub.mkdir(exist_ok=True)
for z in glob.glob("/kaggle/working/sm_*/submission.zip"):
    p = Path(z); shutil.copy(z, sub / f"{p.parent.name}.zip")
r = subprocess.run([sys.executable, "sm_ens.py"] + (["--smoke"] if SMOKE else []), capture_output=True, text=True)
print(r.stdout[-20000:], r.stderr[-5000:] if r.returncode else "")      # stderr only on failure (hides the roberta load report)
print("\n=== official evaluate_phase6.py on each run's held-out predictions ===")
for d in sorted(glob.glob("/kaggle/working/sm_*/dev_truth.json")):
    d = Path(d).parent
    if ("_smoke" in d.name) != SMOKE:
        continue
    r = subprocess.run([sys.executable, "evaluate_phase6.py", "--truth", str(d / "dev_truth.json"), "--prediction", str(d / "dev_prediction.json")],
                       capture_output=True, text=True)
    print(d.name, "|", " | ".join(l for l in r.stdout.split("\n") if l), r.stderr[-300:] if r.returncode else "")
print(sorted(os.listdir(sub)))
print("\n=== SUMMARIES ===")
for lg in sorted(LOG.glob("*.log")):
    for line in open(lg).read().splitlines():
        if line.startswith(("SUMMARY", "KD:", "DEV ", "decode ", "chosen", "reference point", "overlap", "train:", "dev:", "test:",
                            "length model", "novel summary", "AdaLoRA final", "[qlora]", "Phase-1 ", "restored")) \
                or "Traceback" in line or "Error" in line:
            print(lg.stem, "|", line)
print("ALL_DONE")


BERTScore ready 0.9871428608894348
 
GPUs: 2 | queues: {0: [('lora', 'gqa', '', {}), ('qlora', 'retnet', '', {}), ('pfeiffer', 'mla', '', {})], 1: [('lorma', 'encdec', '', {}), ('houlsby', 'encdec', '', {}), ('adalora', 'mla', '', {})]} | final: {0: [('kd', 'gqa', '', {})], 1: [('lorma', 'encdec', '_1chunk', {'K_ENC': 1})]} | smoke: False
[   0.0 min] GPU0 started lora_gqa
[   0.0 min] GPU1 started lorma_encdec
[   0.0 min]   GPU0 lora_gqa: []
[   0.0 min]   GPU1 lorma_encdec: []
[   5.0 min]   GPU0 lora_gqa: ['  ep 1 step 40/1684 summary-token CE 2.7113 | lr x0.99 | 2,366 target tok/s | 3.1 min']
[   5.0 min]   GPU1 lorma_encdec: ['  ep 1 step 40/1684 summary-token CE 3.3053 | lr x0.99 | 2,723 target tok/s | 2.9 min']
[  10.0 min]   GPU0 lora_gqa: ['  ep 1 step 110/1684 summary-token CE 2.5433 | lr x0.95 | 2,300 target tok/s | 8.7 min']
[  10.0 min]   GPU1 lorma_encdec: ['  ep 1 step 110/1684 summary-token CE 3.1729 | lr x0.96 | 2,627 target tok/s | 8.1 min']
[  15.0 min]   GPU0 lora_